1. Introduction and Objective

Performance Report: Pandas vs. Dask vs. Spark
The objective of this work is to compare the efficiency of different data processing libraries using the NY taxi dataset (61MB).

Group: Renata Azevedo Barros, Giuliano and Ni.

+ Starting the reading with some necessary imports:

In [1]:
import dask.dataframe as dd
import pyspark.pandas as ps
import time
import os

# Verificação de segurança: listando arquivos na pasta
print("Arquivos detectados na pasta:", os.listdir())

Arquivos detectados na pasta: ['experimento_1.ipynb', 'yellow_tripdata_2026-01.parquet']


c:\Users\Renata\anaconda3\envs\performance_analysis\lib\site-packages\pyspark\pandas\__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


Task 1: Reading the data

+ First experiment: Reading Data with the Pandas library:

In [9]:
import pandas as pd
import time

# --- TESTE PANDAS (TAREFA 1) ---
print("Iniciando leitura com Pandas...")

start_pandas = time.time()

# Lendo o arquivo parquet
df_pandas = pd.read_parquet('yellow_tripdata_2026-01.parquet')

# Contando as linhas
qtd_linhas = len(df_pandas)

tempo_pandas = time.time() - start_pandas

print(f"Resultado Pandas:")
print(f"- Linhas: {qtd_linhas}")
print(f"- Tempo de execução: {tempo_pandas:.4f} segundos")

Iniciando leitura com Pandas...
Resultado Pandas:
- Linhas: 3724889
- Tempo de execução: 0.3284 segundos


+ Second experiment: Reading Data with the Dask library:

In [2]:
start = time.time()
df_dask = dd.read_parquet('yellow_tripdata_2026-01.parquet')
count = df_dask.shape[0].compute()
print(f"Dask: {count} linhas em {time.time() - start:.2f}s")

Dask: 3724889 linhas em 0.19s


+ Third experiment: Reading data with Spark/Koalas:

In [5]:
import os
from pyspark.sql import SparkSession
import pyspark.pandas as ps
import time

# 1. Pegando o caminho completo do arquivo para não ter erro
caminho_arquivo = os.path.abspath('yellow_tripdata_2026-01.parquet')
print(f"Lendo de: {caminho_arquivo}")

# 2. Iniciando a sessão de forma ultra-estável
spark = SparkSession.builder \
    .master("local[1]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

print("Motor iniciado. Lendo...")

# 3. Teste de Leitura
try:
    start = time.time()
    # Usamos o caminho absoluto com o prefixo 'file:///' que o Windows/Spark exigem
    df_koalas = ps.read_parquet(f"file:///{caminho_arquivo.replace(os.sep, '/')}")
    
    count = len(df_koalas)
    print(f"Sucesso! Koalas: {count} linhas em {time.time() - start:.2f}s")
except Exception as e:
    print(f"Houve um erro na leitura: {e}")

Lendo de: c:\TD\yellow_tripdata_2026-01.parquet


Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.UnsupportedOperationException: getSubject is supported only if a security manager is allowed
	at java.base/javax.security.auth.Subject.getSubject(Subject.java:347)
	at org.apache.hadoop.security.UserGroupInformation.getCurrentUser(UserGroupInformation.java:588)
	at org.apache.spark.util.Utils$.$anonfun$getCurrentUserName$1(Utils.scala:2446)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.util.Utils$.getCurrentUserName(Utils.scala:2446)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:339)
	at org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:59)
	at java.base/jdk.internal.reflect.DirectConstructorHandleAccessor.newInstance(DirectConstructorHandleAccessor.java:62)
	at java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:501)
	at java.base/java.lang.reflect.Constructor.newInstance(Constructor.ja

Durante a Tarefa 1, observou-se que as bibliotecas Pandas e Dask apresentaram alta compatibilidade com o ambiente Windows. O Spark (Koalas), entretanto, apresentou erros de Java Gateway (Py4JJavaError). Isso ocorre porque o Spark depende de uma infraestrutura de suporte (Hadoop Winutils e variáveis de ambiente complexas) que torna sua implementação local em Windows menos eficiente para análises rápidas comparado ao Dask.

In [11]:
import pandas as pd

# 1. Crie um dicionário com os resultados que você anotou
dados_comparativos = {
    'Biblioteca': ['Pandas', 'Dask', 'Spark (Koalas)'],
    'Tempo de Leitura (s)': [0.3284, 0.19, 0.0000],  # <-- SUBSTITUA PELOS SEUS TEMPOS
    'Status': ['Sucesso', 'Sucesso', 'Erro de Ambiente (Java/NumPy)'],
    'Observação': [
        'Eager Execution: Carregamento completo e imediato dos dados na memória.',
        'Lazy Evaluation: Mapeamento rápido de metadados; o processamento real é adiado.',
        'Falha de infraestrutura local (conflito de versões Java/Python).'
    ]
}

# 2. Transforme em um DataFrame do Pandas
df_resumo = pd.DataFrame(dados_comparativos)

# 3. Exiba a tabela formatada
print("TABELA COMPARATIVA - TAREFA 1")
display(df_resumo)

TABELA COMPARATIVA - TAREFA 1


,Biblioteca,Tempo de Leitura (s),Status,Observação
0,Pandas,0.3284,Sucesso,Eager Execution: Carregamento completo e imedi...
1,Dask,0.1900,Sucesso,Lazy Evaluation: Mapeamento rápido de metadado...
2,Spark (Koalas),0.0000,Erro de Ambiente (Java/NumPy),Falha de infraestrutura local (conflito de ver...


* Analysis of Task 1 (Reading)

In Task 1, it was observed that Dask recorded a shorter read time than Pandas (0.19s vs 0.32s). However, this difference does not reflect a superior actual processing speed, but rather the difference between the execution strategies:

Pandas (Eager Evaluation): Performs immediate and complete loading of data into RAM, leaving the DataFrame ready for use.

Dask (Lazy Evaluation): Performs only the mapping of metadata and the file schema (header). The actual data reading is deferred until a calculation operation is requested.

Therefore, the reduced time of Dask in Task 1 is only the cost of initializing the task graph, and not the complete reading of the 61MB of data.

Task 2: Aggregation and Filtering operations.

+ First experiment: Value Counts. Count how many trips occurred by payment type.

    With the Pandas library:

In [12]:
start_p2 = time.time()
counts_pandas = df_pandas['VendorID'].value_counts()
print(f"Tempo Value Counts (Pandas): {time.time() - start_p2:.4f}s")
print(counts_pandas)

Tempo Value Counts (Pandas): 0.0208s
VendorID
2    2965742
1     710425
7      44705
6       4017
Name: count, dtype: int64


    With the Dask:

In [13]:
start_d2 = time.time()
# Lembre-se: no Dask precisamos do .compute() para ele agir
counts_dask = df_dask['VendorID'].value_counts().compute()
print(f"Tempo Value Counts (Dask): {time.time() - start_d2:.4f}s")
print(counts_dask)

Tempo Value Counts (Dask): 0.1814s
VendorID
1     710425
2    2965742
6       4017
7      44705
Name: count, dtype: int64


+ Second experiment: GroupBy. Let's calculate the average tariff value for each type of supplier.
    
    With the Pandas library:

In [15]:
start_p3 = time.time()
mean_pandas = df_pandas.groupby('VendorID')['fare_amount'].mean()
print(f"Tempo GroupBy (Pandas): {time.time() - start_p3:.4f}s")

print("\n--- Média por Fornecedor (Pandas) ---")
print(mean_pandas)

Tempo GroupBy (Pandas): 0.0404s

--- Média por Fornecedor (Pandas) ---
VendorID
1    20.185694
2    21.046596
6     2.720065
7    16.181983
Name: fare_amount, dtype: float64


With the Dask:

In [16]:
start_d3 = time.time()
mean_dask = df_dask.groupby('VendorID')['fare_amount'].mean().compute()
print(f"Tempo GroupBy (Dask): {time.time() - start_d3:.4f}s")

# Exibindo o resultado
print("\n--- Média por Fornecedor (Dask) ---")
print(mean_dask)

Tempo GroupBy (Dask): 0.2666s

--- Média por Fornecedor (Dask) ---
VendorID
1    20.185694
2    21.046596
6     2.720065
7    16.181983
Name: fare_amount, dtype: float64


+ Third experiment: Complex Filtration. We will filter only the rides that had more than 2 passengers and calculate the average distance traveled for that specific group.
    
    With the Pandas library:

In [20]:

start_p4 = time.time()
# Filtramos as linhas onde passageiros > 2 e pegamos a média da coluna trip_distance
filtro_p = df_pandas[df_pandas['passenger_count'] > 2]['trip_distance'].mean()
tempo_p = time.time() - start_p4

print(f"Tempo Filtro (Pandas): {tempo_p:.4f}s")
print(f"Média de distância (Passageiros > 2) no Pandas: {filtro_p:.2f} milhas")

Tempo Filtro (Pandas): 0.0451s
Média de distância (Passageiros > 2) no Pandas: 3.76 milhas


With the Dask:

In [21]:
start_d4 = time.time()
# O processo é idêntico, mas o .compute() é essencial para disparar o cálculo
filtro_d = df_dask[df_dask['passenger_count'] > 2]['trip_distance'].mean().compute()
tempo_d = time.time() - start_d4

print(f"Tempo Filtro (Dask): {tempo_d:.4f}s")
print(f"Média de distância (Passageiros > 2) no Dask: {filtro_d:.2f} milhas")

Tempo Filtro (Dask): 0.2418s
Média de distância (Passageiros > 2) no Dask: 3.76 milhas


In [22]:
import pandas as pd
import dask.dataframe as dd
import time

def tarefa_2_automatizada(df_p, df_d):
    resultados = []

    # --- 1. Operação: Value Counts (VendorID) ---
    # Pandas
    start = time.time()
    df_p['VendorID'].value_counts()
    t_p_vc = time.time() - start

    # Dask
    start = time.time()
    df_d['VendorID'].value_counts().compute()
    t_d_vc = time.time() - start
    
    resultados.append(['Value Counts', t_p_vc, t_d_vc])

    # --- 2. Operação: GroupBy Mean (fare_amount) ---
    # Pandas
    start = time.time()
    df_p.groupby('VendorID')['fare_amount'].mean()
    t_p_gb = time.time() - start

    # Dask
    start = time.time()
    df_d.groupby('VendorID')['fare_amount'].mean().compute()
    t_d_gb = time.time() - start
    
    resultados.append(['GroupBy Mean', t_p_gb, t_d_gb])

    # --- 3. Operação: Filtragem (Passageiros > 2) ---
    # Pandas
    start = time.time()
    df_p[df_p['passenger_count'] > 2]['trip_distance'].mean()
    t_p_fi = time.time() - start

    # Dask
    start = time.time()
    df_d[df_d['passenger_count'] > 2]['trip_distance'].mean().compute()
    t_d_fi = time.time() - start
    
    resultados.append(['Filtragem + Média', t_p_fi, t_d_fi])

    # Criando a tabela final
    df_final = pd.DataFrame(resultados, columns=['Operação', 'Tempo Pandas (s)', 'Tempo Dask (s)'])
    
    # Adicionando coluna de quem venceu
    df_final['Vencedor'] = df_final.apply(lambda x: 'Pandas' if x['Tempo Pandas (s)'] < x['Tempo Dask (s)'] else 'Dask', axis=1)
    
    return df_final

# EXECUTANDO A TAREFA
tabela_tarefa_2 = tarefa_2_automatizada(df_pandas, df_dask)

print("### RESULTADOS AUTOMÁTICOS DA TAREFA 2 ###")
display(tabela_tarefa_2)

### RESULTADOS AUTOMÁTICOS DA TAREFA 2 ###


,Operação,Tempo Pandas (s),Tempo Dask (s),Vencedor
0,Value Counts,0.010678,0.080139,Pandas
1,GroupBy Mean,0.039452,0.234897,Pandas
2,Filtragem + Média,0.044123,0.198536,Pandas


* Analysis of Task 2 (Operations)

As predicted, when we performed aggregation operations in Task 2, the scenario reversed. Since Pandas had already loaded the data in Task 1, its response was almost instantaneous, while Dask finally needed to read the file to perform the calculations, confirming its lazy nature.

Task 3: Join: The Databricks benchmark performs a join to translate IDs into location names

+ First experiment: With the Pandas library:

In [24]:

# 1. Carregar e concatenar os dois meses (Escalabilidade)
df_jan = pd.read_parquet('yellow_tripdata_2026-01.parquet')
df_fev = pd.read_parquet('yellow_tripdata_2026-02.parquet')
df_viagens_p = pd.concat([df_jan, df_fev])

# 2. Criar a tabela de referência de Zonas
data_zones = {'PULocationID': [1, 2, 4, 132, 138], 
              'ZoneName': ['Newark Airport', 'Jamaica Bay', 'Alphabet City', 'JFK Airport', 'LaGuardia Airport']}
df_zones_p = pd.DataFrame(data_zones)

# 3. Operação de Join (Merge)
start_p = time.time()
# how='left' garante que não perdemos viagens que não tenham correspondência na tabela de zonas
df_final_p = df_viagens_p.merge(df_zones_p, on='PULocationID', how='left')
tempo_p = time.time() - start_p

print(f"Tempo Join Pandas (2 meses): {tempo_p:.4f}s")
print(df_final_p[['PULocationID', 'ZoneName', 'fare_amount']].head())

Tempo Join Pandas (2 meses): 1.8118s
   PULocationID ZoneName  fare_amount
0           239      NaN          7.2
1           163      NaN          7.9
2            43      NaN         10.7
3           142      NaN         38.7
4            88      NaN         13.5


With the Dask:

In [25]:
import dask.dataframe as dd
import time

# 1. Carregar os dois meses de forma eficiente (Escalabilidade)
# O Dask permite usar uma lista de ficheiros diretamente
df_viagens_d = dd.read_parquet(['yellow_tripdata_2026-01.parquet', 'yellow_tripdata_2026-02.parquet'])

# 2. Criar a tabela de referência (Zonas) no Dask
# Nota: Convertemos de Pandas para Dask
df_zones_d = dd.from_pandas(pd.DataFrame(data_zones), npartitions=1)

# 3. Operação de Join (Merge)
start_d = time.time()
# O Merge no Dask é "Lazy" (preguiçoso), ele só cria o plano de execução
df_final_d = df_viagens_d.merge(df_zones_d, on='PULocationID', how='left')

# O .compute() é que executa o Join de facto e traz o resultado para a memória
resultado_d = df_final_d.head() 
tempo_d = time.time() - start_d

print(f"Tempo Join Dask (2 meses): {tempo_d:.4f}s")
print(resultado_d[['PULocationID', 'ZoneName', 'fare_amount']])

c:\Users\Renata\anaconda3\envs\performance_analysis\lib\site-packages\dask\dataframe\multi.py:520: UserWarning: Merging dataframes with merge column data type mismatches: 
+----------------------------------+------------+-------------+
| Merge columns                    | left dtype | right dtype |
+----------------------------------+------------+-------------+
| ('PULocationID', 'PULocationID') | int32      | int64       |
+----------------------------------+------------+-------------+
Cast dtypes explicitly to avoid unexpected results.
  warnings.warn(


Tempo Join Dask (2 meses): 1.5342s
   PULocationID ZoneName  fare_amount
0           239     <NA>          7.2
1           163     <NA>          7.9
2            43     <NA>         10.7
3           142     <NA>         38.7
4            88     <NA>         13.5


Task 4: Arithmetic Calculation (Cálculo)

    + First experiment: With the Pandas library:

In [26]:
start = time.time()
df_pandas['tip_per_passenger'] = df_pandas['tip_amount'] / df_pandas['passenger_count']
print(f"Tempo Cálculo Pandas: {time.time() - start:.4f}s")

Tempo Cálculo Pandas: 0.0589s


+ First experiment: With the Dask:

In [27]:
start = time.time()
df_dask['tip_per_passenger'] = df_dask['tip_amount'] / df_dask['passenger_count']
# O Dask só calcula de fato no próximo compute() ou ao salvar
print(f"Tempo Cálculo Dask (Lazy): {time.time() - start:.4f}s")

Tempo Cálculo Dask (Lazy): 0.0050s


Task 5: Writing

    + First experiment: With the Pandas library:

In [28]:
start = time.time()
df_pandas.to_parquet('resultado_pandas.parquet')
print(f"Tempo Escrita Pandas: {time.time() - start:.4f}s")

Tempo Escrita Pandas: 1.5492s


+ Second experiment: With the Dask:

In [29]:
start = time.time()
# O Dask salva em múltiplos arquivos (partições)
df_dask.to_parquet('resultado_dask_folder', engine='pyarrow')
print(f"Tempo Escrita Dask: {time.time() - start:.4f}s")

Tempo Escrita Dask: 4.1810s


2. Methodology

Environment and Challenges
Hardware:? RAM

Technical Challenges: During execution, the Spark/Koalas library showed incompatibility with the local environment (Windows + Java Gateway) and conflicts with NumPy version 2.0. Therefore, the performance analysis focused on a detailed comparison between Pandas (Eager) and Dask (Lazy).

In [30]:
import pandas as pd

# Substitua os valores abaixo pelos seus tempos reais
resultados_benchmark = {
    'Etapa do Benchmark': [
        '1. Data Loading', 
        '2. Value Counts', 
        '3. GroupBy Mean', 
        '4. Filtering', 
        '5. Join', 
        '6. Arithmetic Calculation', 
        '7. Writing (Persistência)'
    ],
    'Pandas (s)': [0.3284, 0.0210, 0.0450, 0.0320, 0.0510, 0.0150, 0.2100], # <-- EXEMPLOS
    'Dask (s)':   [0.1900, 0.3540, 0.4210, 0.3880, 0.5800, 0.0050, 0.8500]  # <-- EXEMPLOS
}

df_benchmark = pd.DataFrame(resultados_benchmark)

# Cálculo de Speedup (Quantas vezes o Pandas foi mais rápido)
df_benchmark['Diferença'] = df_benchmark['Dask (s)'] / df_benchmark['Pandas (s)']

print("### BENCHMARK FINAL: PANDAS vs DASK (Dataset 61MB) ###")
display(df_benchmark)

### BENCHMARK FINAL: PANDAS vs DASK (Dataset 61MB) ###


,Etapa do Benchmark,Pandas (s),Dask (s),Diferença
0,1. Data Loading,0.3284,0.190,0.578563
1,2. Value Counts,0.0210,0.354,16.857143
2,3. GroupBy Mean,0.0450,0.421,9.355556
3,4. Filtering,0.0320,0.388,12.125000
4,5. Join,0.0510,0.580,11.372549
5,6. Arithmetic Calculation,0.0150,0.005,0.333333
6,7. Writing (Persistência),0.2100,0.850,4.047619


3. Análise de Resultados

